# 数据类与三种构建器

## 什么是「数据类」

「数据类」是一种编码模式：**只作为字段集合、几乎没有额外功能的简单类**。其他语言里的对等物如 Java 的 POJO、Kotlin 的 data class。Python 里有三个专门的类构建器：

| 构建器 | 登场版本 | 形态 | 特点 |
|--------|:---:|------|------|
| `collections.namedtuple` | 2.6 | 工厂函数 | 最简单，构建 tuple 子类 |
| `typing.NamedTuple` | 3.5（3.6 起支持 class 语法） | 类语法 | 带类型提示的具名元组 |
| `@dataclasses.dataclass` | 3.7 | 类装饰器 | 功能最全、定制性最强 |

## 朴素类的问题

不用任何构建器，手写一个坐标类：

In [23]:
class Coordinate:
    def __init__(self,lat,lon):
        self.lat=lat
        self.lon=lon

两个问题：`__init__` 样板代码每个属性要写 3 次（形参、赋值、self.xxx）；从 object 继承的功能非常粗糙：

In [24]:
moscow=Coordinate(55.76,37.62)
moscow # __repr__无用

In [25]:
location=Coordinate(55.76,37.62)
location==moscow    # __eq__比较的是对象ID

False

In [26]:
# 相比较内容必须
(location.lat,location.lon)==(moscow.lat,moscow.lon)

True

## 三种构造器一瞥

### collections.namedtuple:工厂函数

In [27]:
from collections import namedtuple

Coordinate=namedtuple('Coordinate','lat lon')
issubclass(Coordinate,tuple)
moscow=Coordinate(55.76,37.62)
print(moscow)
moscow==Coordinate(lat=55.76,lon=37.62)


Coordinate(lat=55.76, lon=37.62)


True

### typing.NamedTuple:支持类型提示

In [28]:
import typing
Coordinate=typing.NamedTuple('Coordinate',[('lat',float),('lon',float)])
typing.get_type_hints(Coordinate)

{'lat': float, 'lon': float}

In [29]:
from typing import NamedTuple

class Coordinate(NamedTuple):
    lat: float
    lon: float

    def __str__(self):
        ns = 'N' if self.lat >= 0 else 'S'
        we = 'E' if self.lon >= 0 else 'W'
        return f'{abs(self.lat):.1f}°{ns}, {abs(self.lon):.1f}°{we}'

moscow=Coordinate(55.76,37.62)
print(moscow)

55.8°N, 37.6°E


### @dataclass:装饰器

In [30]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinate:
    lat:float
    lon:float

moscow=Coordinate(55.76,37.62)
print(moscow)

location=Coordinate(55.76,37.62)
moscow==location

Coordinate(lat=55.76, lon=37.62)


True

# collections.namedtuple：最经典的具名元组

## 定义与使用

namedtuple(类名, 字段名列表) 是工厂函数，用于构建带字段名、类名和详细 __repr__ 的 tuple 子类：

In [31]:
from collections import namedtuple
City = namedtuple('City', 'name country population coordinates')   # ①
tokyo = City('Tokyo', 'JP', 36.933, (35.689722, 139.691667))       # ②
print(tokyo)
print(tokyo.population)        # ③ 按字段名访问
print(tokyo.coordinates)
print(tokyo[1])               # ③ 也能按位置访问（本质是 tuple）

City(name='Tokyo', country='JP', population=36.933, coordinates=(35.689722, 139.691667))
36.933
(35.689722, 139.691667)
JP


- ① 字段名列表可以是「空格分隔的单个字符串」或「可生成多个字符串的可迭代对象」；
- ② 字段值必须逐个作为位置参数传入（对比：tuple 构造函数只接受单个可迭代对象）；
- ③ 可按字段名或位置访问字段。

> 两个实用事实：具名元组实例占用的内存与 tuple 完全相同（字段名存储在类上，不在实例里）；作为 tuple 子类，它继承了 __eq__、__lt__ 等比较方法，所以由 City 实例构成的列表可以直接 sorted()。

## 属性与方法

In [32]:
print(f'类属性：字段名元组：{City._fields}')
from collections import namedtuple

# 报错原因：前面章节里 Coordinate 最后一次被定义成了 @dataclass（普通对象），
# 而 City._asdict() 只展开 City 这一层，嵌套的 coordinates 仍是 Coordinate 实例，
# json.dumps 遇到不认识的类型就抛 TypeError: Object of type Coordinate is not JSON serializable。
# 本节讨论具名元组，因此恢复书中的定义：Coordinate 是 tuple 子类，可直接编码成 JSON 数组。
Coordinate = namedtuple('Coordinate', 'lat lon')
delhi_data=('Delhi NCR','IN',21.935,Coordinate(28.613889,77.208889))
delhi=City._make(delhi_data) # 类方法：从可迭代对象创建实例，等价于 City(*delhi_data)
print(f'实例方法：转成dict （键=字段名，值=字段值）\n{delhi._asdict()}')

import json
json.dumps(delhi._asdict())

类属性：字段名元组：('name', 'country', 'population', 'coordinates')
实例方法：转成dict （键=字段名，值=字段值）
{'name': 'Delhi NCR', 'country': 'IN', 'population': 21.935, 'coordinates': Coordinate(lat=28.613889, lon=77.208889)}


'{"name": "Delhi NCR", "country": "IN", "population": 21.935, "coordinates": [28.613889, 77.208889]}'

> 版本细节：_asdict() 在 Python 3.7 及以前返回 OrderedDict；3.8 起返回普通 dict（因为 3.7 起 dict 保证插入顺序）。必须用 OrderedDict 时，自行 OrderedDict(x._asdict())。